# Tiền xử lý dữ liệu VnExpress — HARNN

**Input:** `raw_data.json` — dữ liệu crawl thô  
**Output:** `dataset.json` · `vocab.json` · `label_map.json`

| Cell | Nhiệm vụ |
|------|----------|
| 1 | Cấu hình đường dẫn & tham số |
| 2 | HIERARCHY — cây nhãn cố định |
| 3 | Normalize & gộp nhãn L2/L3 |
| 4 | Hàm text processing |
| 5 | Load → tokenize → encode label |
| 6 | Build vocab → lưu file |
| 7 | Kiểm tra kết quả |


## Cell 1 — Cấu hình


In [1]:
import json, re
from collections import Counter, defaultdict
from pathlib import Path

# ── Đường dẫn ─────────────────────────────────────────────────────────────
RAW_FILE = r'C:\Users\Admin\Documents\nlp\harnn_project\data\raw\raw_data.json' 

PROJECT_DIR = Path(r'C:\Users\Admin\Documents\nlp\harnn_project')
PROCESS_DIR = PROJECT_DIR / 'data' / 'process_data'
OUT_DATASET = PROCESS_DIR / 'dataset.json'
OUT_VOCAB = PROCESS_DIR / 'vocab.json'
OUT_LABEL_MAP = PROCESS_DIR / 'label_map.json'
STOPWORDS_FILE = PROJECT_DIR / 'data' / 'dictionary' / 'vietnamese-stopwords.txt'

# ── Tham số ───────────────────────────────────────────────────────────────
MIN_TOKENS      = 20   # bỏ bài có ít hơn N token
VOCAB_MIN_COUNT = 3    # bỏ từ xuất hiện ít hơn N lần

# Tạo thư mục output tự động: data/process_data
PROCESS_DIR.mkdir(parents=True, exist_ok=True)

print('Config OK')
print(f'Input (absolute): {RAW_FILE}')
print(f'Output folder   : {PROCESS_DIR}')


Config OK
Input (absolute): C:\Users\Admin\Documents\nlp\harnn_project\data\raw\raw_data.json
Output folder   : C:\Users\Admin\Documents\nlp\harnn_project\data\process_data


## Cell 2 — HIERARCHY
Cây nhãn **cố định** — index không đổi giữa các lần chạy, đúng tinh thần paper.


In [2]:
# HIERARCHY[l1][l2] = [l3, ...]  |  list rỗng = không có L3
HIERARCHY = {
    'Thời sự': {
        'Chính trị': [], 'Dân sinh': [], 'Giao thông': [], 'Diễn đàn': [],
    },
    'Thế giới': {
        'Quân sự': [], 'Quốc tế': [], 'Tư liệu': [], 'Phân tích': [],
    },
    'Kinh doanh': {
        'Thị trường': [], 'Hàng hoá': [], 'Doanh nghiệp': [],
        'Tiêu dùng': [],
    },
    'Khoa học': {
        'Vũ trụ': [],
    },
    'Giải trí': {
        'Giải sao':  ['Sao đẹp - Sao xấu', 'Chuyện màn ảnh'],
        'Phim':      [],
        'Sách':      [],
        'Thời trang':['Lăng mốt'],
        'Nhạc':      ['Lăng nhạc'],
        'Sân khấu - Mỹ thuật': [],
    },
    'Thể thao': {
        'Bóng đá':      ['Champions League', 'Trong nước', 'Thế giới',
                         'La Liga', 'Các giải khác'],
        'Tennis':       [],
        'Marathon':     [],
        'Các môn khác': [],
    },
    'Pháp luật': {
        'Hồ sơ phá án': [], 'Tư vấn': [],
    },
    'Giáo dục': {
        'Tuyển sinh': ['Đại học'], 'Du học': [], 'Chân dung': [],
    },
    'Sức khỏe': {
        'Các bệnh': [
            'Nhân khoa', 'Tim mạch', 'Nội tiết', 'Thần kinh', 'Tiêu hóa',
            'Da liễu', 'Hô hấp', 'Sản phụ khoa', 'Cơ xương khớp',
            'Ung thư', 'Tai mũi họng', 'Tiết niệu - Nam học', 'Nhi - Sơ sinh',
        ],
        'Sống khỏe':          ['Dinh dưỡng', 'Hỏi đáp', 'Tư vấn'],
        'Tin tức sức khỏe':   [],
    },
    'Đời sống': {
        'Nhịp sống':  ['Lăng văn', 'Bạn đọc viết'],
        'Ẩm thực':    [],
        'Tổ ấm':      [],
    },
    'Du lịch': {
        'Điểm đến': [], 'Ẩm thực': [], 'Dấu chân': [],
    },
    'Số hóa': {
        'Thiết bị':         [],
        'AI':               [],
        'Chuyển đổi số':    [],
        'Tin tức số hóa':   [],
    },
    'Xe': {
        'Cầm lái':       ['Kỹ năng lái', 'Tình huống', 'Đánh giá xe'],
        'Thị trường xe': [],
        'Tin tức xe':    [],
    },

    'Thời sự':    {'Chính trị':[], 'Dân sinh':[], 'Giao thông':[], 'Diễn đàn':[]},
    'Thế giới':   {'Quân sự':[], 'Quốc tế':[], 'Tư liệu':[], 'Phân tích':[]},
    'Kinh doanh': {'Thị trường':[], 'Hàng hoá':[], 'Doanh nghiệp':[],
                   'Tiêu dùng':[], 'Vĩ mô':[]},
    'Khoa học':   {'Vũ trụ':[], 'Thế giới tự nhiên':[]},
    'Giải trí':   {'Giải sao':[], 'Phim':[], 'Sách':[], 'Thời trang':[],
                   'Nhạc':[], 'Hậu trường':[]},
    'Thể thao':   {
        'Bóng đá':     ['Champions League', 'Trong nước', 'Thế giới'],
        'Tennis':      [],
        'Marathon':    [],
        'Các môn khác':[],
    },
    'Pháp luật':  {'Hồ sơ phá án':[], 'Tư vấn':[]},
    'Giáo dục':   {'Tuyển sinh':[], 'Du học':[], 'Chân dung':[]},
    'Sức khỏe':   {
        'Các bệnh':    ['Nhân khoa', 'Tim mạch', 'Nội tiết', 'Thần kinh',
                        'Tiêu hóa', 'Da liễu', 'Hô hấp', 'Sản phụ khoa',
                        'Cơ xương khớp', 'Ung thư', 'Tai mũi họng'],
        'Sống khỏe':   ['Dinh dưỡng', 'Tư vấn', 'Vaccine'],
        'Tin tức sức khỏe': [],
    },
    'Đời sống':   {'Nhịp sống':[], 'Cuộc sống đó đây':[], 'Tổ ấm':[],
                   'Bài học sống':[]},
    'Du lịch':    {'Điểm đến':[], 'Ẩm thực':[], 'Dấu chân':[]},
    'Số hóa':     {'Thiết bị':[], 'AI':[], 'Chuyển đổi số':[],
                   'Tin tức số hóa':[]},
    'Xe':         {
        'Cầm lái':  ['Kỹ năng lái', 'Đánh giá xe'],
        'Thị trường xe': [],
        'Tin tức xe':    [],
    },
}

# Build lookup tables
L1_SET, L2_SET, L3_SET = set(), set(), set()
L2_TO_L1, L3_TO_L2 = {}, {}

for l1, subs in HIERARCHY.items():
    L1_SET.add(l1)
    for l2, l3s in subs.items():
        L2_SET.add(l2);  L2_TO_L1[l2] = l1
        for l3 in l3s:
            L3_SET.add(l3);  L3_TO_L2[l3] = l2

LABEL_MAP = {
    'l1': {lb: i for i, lb in enumerate(sorted(L1_SET))},
    'l2': {lb: i for i, lb in enumerate(sorted(L2_SET))},
    'l3': {lb: i for i, lb in enumerate(sorted(L3_SET))},
}

print(f'L1: {len(LABEL_MAP["l1"])}  L2: {len(LABEL_MAP["l2"])}  L3: {len(LABEL_MAP["l3"])}')


L1: 13  L2: 47  L3: 19


## Cell 3 — Normalize & gộp nhãn
Chuẩn hóa chữ hoa/thường, gộp nhãn trùng nghĩa, loại nhãn sự kiện/format.


In [ ]:
NORMALIZE = {
    'cooking':                   'Ẩm thực',
    'Cooking':                   'Ẩm thực',
    'Khoa học công nghệ':        'Số hóa',
    'Chính trị & chính sách':    'Chính trị',
    'Chính sách':                'Chính trị',
    # 'Sân khấu - Mỹ thuật' giữ nguyên, đã có trong HIERARCHY
}

L2_MERGE = {
    # Giải đấu bóng đá → Bóng đá
    'Ngoại hạng Anh': 'Bóng đá', 'V-League':  'Bóng đá',
    'Bundesliga':     'Bóng đá', 'La Liga':   'Bóng đá',
    'Serie A':        'Bóng đá', 'Europa League': 'Bóng đá',

    # Gộp nhãn trùng nghĩa
    'Thế giới tự nhiên':  'Vũ trụ',
    'Bắc Mỹ':             'Quốc tế',
    'Phân tích':          'Tư liệu',
    'Cuộc sống đó đây':   'Nhịp sống',
    'Bài học sống':       'Nhịp sống',
    'Tổ ấm':              'Nhịp sống',      # ← giữ vì trong HIERARCHY
    'Hậu trường':         'Giải sao',
    'Tiêu dùng':          'Hàng hoá',
    'Dân sự':             'Hồ sơ phá án',
    'Diễn đàn':           'Chính trị',
    'Dấu chân':           'Điểm đến',
    'Sân khấu & Mỹ thuật':'Sân khấu - Mỹ thuật',  # normalize & → -
    'Các môn khác':       '',               # DROP
}

L2_DROP = {
    'Tin tức',   # xử lý riêng qua TIN_TUC_MAP
    'Video', 'Ảnh', 'Podcast',
    'Trắc nghiệm',              # format quiz, không phải category
    'Mekong', 'Xe điện', 'Chứng khoán', 'Ebank',
    'Netzero', 'Kỳ nguyên mới', 'Quỹ hy vọng',
    'Của sổ tri thức', 'Học tiếng Anh', 'Đổi mới sáng tạo', 'Việc làm',
    'Bảo vệ trọn vẹn từng khoảnh khắc', 'Bầu cử Đại biểu Quốc hội khóa 16',
    'Dự án', 'Kun Marathon', 'Giáo dục 4.0', 'Người Việt 5 châu',
    'Kinh tế vùng', 'Tiền của tôi', 'Khoa học trong nước',
    'Khí mát lạnh, sống trọn ven', 'Nội trợ',
    'GameVerse 2026', 'GameVerse 2025', 'Tương thuật', 'esport', 'Bảo hiểm',
    'Sống tinh tế trong thầm lặng',
    # 'Vaccine' ← xóa dòng này
}

# Vấn đề 6 đã sửa: thêm đủ domain
TIN_TUC_MAP = {
    'Sức khỏe':  'Tin tức sức khỏe',
    'Xe':        'Tin tức xe',
    'Số hóa':    'Tin tức số hóa',
    'Thể thao':  '',
    'Giáo dục':  '',
    'Kinh doanh':'',
    'Thời sự':   '',
    'Thế giới':  '',
    'Du lịch':   '',
    'Đời sống':  '',
    'Pháp luật': '',
    'Giải trí':  '',
    'Khoa học':  '',
}

L3_MERGE = {
    # Giải trí
    'Sao đẹp - Sao xấu': 'Giải sao',
    'Chuyện màn ảnh':     'Phim',
    'Lăng nhạc':          'Nhạc',
    'Lăng mốt':           'Thời trang',
    'Bạn đọc viết':       'Nhịp sống',
    'Lăng văn':           'Nhịp sống',

    # Tư vấn / hỏi đáp
    'Hỏi đáp':   'Tư vấn',
    'Y kiến':    'Tư vấn',
    'meo-tu-van':'Tư vấn',

    # Thế giới
    'Phân tích': 'Tư liệu',

    # Thể thao — Vấn đề 3+5 đã sửa
    'Các giải khác': 'Các giải khác',  # giữ nguyên, có trong HIERARCHY
    'Trong nước':    'Trong nước',     # giữ nguyên
    'Thế giới':      'Thế giới',       # giữ nguyên (L3 của Bóng đá)
    'La Liga':       'Các giải khác',  # gộp vào

    # Sức khỏe
    'Tiết niệu - Nam học': 'Tiết niệu - Nam học',
    'Nhi - Sơ sinh':       'Nhi - Sơ sinh',
    'Khỏe đẹp':            'Dinh dưỡng',

    # Xe
    'Tình huống':    'Kỹ năng lái',
    'Luật giao thông':'Kỹ năng lái',
    'Chăm sóc xe':   'Kỹ năng lái',
    'Kinh nghiệm':   'Kỹ năng lái',

    # Giáo dục
    'Lớp 10':        'Đại học',   # gộp vào Tuyển sinh > Đại học

    # DROP qua MERGE trỏ ''
    'Thị trường': '', 'Giao thông': '', 'Sản phẩm': '',
    'Diễn đàn':   '', 'Bóng đá':   '',

    'Tin tức':    '',   # trùng tên, DROP
    'Quốc tế':    '',   # trùng tên L2, DROP
    'Nhịp sống số':'Nhịp sống',
}

L3_DROP = {
    'Video', 'Ảnh',
    'Vô thuật', 'Tương thuật',
    'Kun Marathon Hồ Chí Minh', 'Sene A',
    'Thi tốt nghiệp THPT', 'Câu chuyện bảo hiểm',
    'Thúc đẩy Khoa học Công nghệ', 'Cẩm nang đầu tư F0',
    'Game talk', 'VMC', 'Tin dự án',
    'Hành trình kiêu dũng', 'Doanh nghiệp xanh', 'Doanh nhân',
    'Chuyển đổi số', 'Đầu tư', 'Đất đai', 'Golf', 'Làm đẹp', 'Tác giả',
    'Điểm sách', 'Đạt đến', 'Điểm phim', 'Dấu ấn thương hiệu', 'Nhân sự',
    'Bình luận', 'Chân dung', 'Hà - Đáp', 'Sáng kiến', 'Sống bền vững',
    'Ứng dụng', 'Hình sự', 'Chính sách', 'Trắc nghiệm', 'Thi trường',
    'Hiểm muộn', 'Ngân hàng', 'Thiết bị', 'Bộ sưu tập',
    'Giáo dục phổ thông', 'Các môn thể thao khác', 'Dua xe', 'Cờ vua',
    'Việt Nam',    
    'Của sổ tri thức', 'Lớp 10',
    'Chăm sóc xe', 'Sân phụ khoa',  
    'Video','Ảnh','Vô thuật','Tương thuật',
    'Kun Marathon Hồ Chí Minh','Sene A','Lớp 10','Thi tốt nghiệp THPT',
    'Câu chuyện bảo hiểm','Thúc đẩy Khoa học Công nghệ','Cẩm nang đầu tư F0',
    'Game talk','VMC','Tin dự án','Hành trình kiêu dũng',
    'Doanh nghiệp xanh','Doanh nhân',
    'Chuyển đổi số','Đầu tư','Đất đai','Golf','Làm đẹp','Tác giả',
    'Điểm sách','Đạt đến','Điểm phim','Dấu ấn thương hiệu','Nhân sự',
    'Bình luận','Chân dung','Hà - Đáp','Sáng kiến','Sống bền vững',
    'Ứng dụng','Hình sự','Chính sách','Trắc nghiệm','Thi trường',
    'Hiểm muộn','Ngân hàng','Thiết bị','Bộ sưu tập',
    'Giáo dục phổ thông','Các môn thể thao khác','Dua xe','Cờ vua',
}


def resolve_labels(l1: str, l2: str, l3: str) -> tuple[str, str, str]:
    """Normalize + merge + validate 3 nhãn, trả về tuple (l1, l2, l3) sạch."""
    # Normalize
    l1 = NORMALIZE.get(l1.strip(), l1.strip())
    l2 = NORMALIZE.get(l2.strip(), l2.strip())
    l3 = NORMALIZE.get(l3.strip(), l3.strip())

    # Validate L1
    if l1 not in L1_SET:
        return '', '', ''

    # Xử lý L2
    if l2 == 'Tin tức':
        l2 = TIN_TUC_MAP.get(l1, '')
    elif l2 in L2_DROP:
        l2 = ''
    else:
        l2 = L2_MERGE.get(l2, l2)
    if l2 not in L2_SET:
        l2 = ''

    # Xử lý L3
    if l3 in L3_DROP:
        l3 = ''
    else:
        l3 = L3_MERGE.get(l3, l3)
    if l3 not in L3_SET:
        l3 = ''

    # Enforce parent-child: có L3 → bổ sung L2 cha nếu thiếu
    if l3 and not l2:
        l2 = L3_TO_L2.get(l3, '')

    return l1, l2, l3


# Test nhanh
cases = [
    ('Thời sự',  'Tin tức', ''),
    ('Sức khỏe', 'Tin tức', ''),
    ('Thể thao', 'Ngoại hạng Anh', ''),
    ('',         'Bóng đá', ''),
    ('Xe',       'Cầm lái', 'Tình huống'),
]
for l1, l2, l3 in cases:
    r = resolve_labels(l1, l2, l3)
    print(f'  ({l1!r:<12}, {l2!r:<18}, {l3!r:<12}) → {r}')


  ('Thời sự'   , 'Tin tức'         , ''          ) → ('Thời sự', '', '')
  ('Sức khỏe'  , 'Tin tức'         , ''          ) → ('Sức khỏe', 'Tin tức sức khỏe', '')
  ('Thể thao'  , 'Ngoại hạng Anh'  , ''          ) → ('Thể thao', 'Bóng đá', '')
  (''          , 'Bóng đá'         , ''          ) → ('', '', '')
  ('Xe'        , 'Cầm lái'         , 'Tình huống') → ('Xe', 'Cầm lái', 'Kỹ năng lái')


## Cell 4 — Text processing


In [4]:
with open(STOPWORDS_FILE, encoding='utf-8') as f:
    STOPWORDS = {line.strip() for line in f if line.strip()}
# Chuẩn hóa: file có thể dùng khoảng trắng, underthesea dùng gạch dưới
STOPWORDS |= {w.replace(' ', '_') for w in STOPWORDS}
print(f'Loaded {len(STOPWORDS)} stopwords')


def clean(text: str) -> str:
    text = re.sub(r'https?://\S+', ' ', text)
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\d+', ' ', text)
    return re.sub(r'\s+', ' ', text).strip().lower()


def tokenize(text: str) -> list[str]:
    try:
        from underthesea import word_tokenize
        tokens = word_tokenize(clean(text), format='text').split()
    except ImportError:
        tokens = clean(text).split()
    return [t for t in tokens if t not in STOPWORDS and len(t) > 1]


def multihot(label: str, idx_map: dict) -> list[int]:
    vec = [0] * len(idx_map)
    if label and label in idx_map:
        vec[idx_map[label]] = 1
    return vec


# Test
sample = 'Cứu được người đuối nước khi livestream lúc 12h30 ngày 17/3'
print(f'Input : {sample}')
print(f'Tokens: {tokenize(sample)}')


Loaded 3513 stopwords
Input : Cứu được người đuối nước khi livestream lúc 12h30 ngày 17/3


c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Tokens: ['cứu', 'đuối', 'livestream']


## Cell 5 — Load → tokenize → encode label


In [5]:
with open(RAW_FILE, encoding='utf-8') as f:
    raw = json.load(f)
print(f'Loaded {len(raw):,} bài thô')

articles = []
skipped  = 0

for i, a in enumerate(raw):
    # Resolve nhãn
    l1, l2, l3 = resolve_labels(
        a.get('label_l1', ''),
        a.get('label_l2', ''),
        a.get('label_l3', ''),
    )
    if not l1:
        skipped += 1
        continue

    # Tokenize
    tokens = tokenize(a.get('title', '') + ' ' + a.get('content', ''))
    if len(tokens) < MIN_TOKENS:
        skipped += 1
        continue

    articles.append({
        'url':           a.get('url', ''),
        'title':         a.get('title', ''),
        'tokens':        tokens,
        'labels_l1':     [l1],
        'labels_l2':     [l2] if l2 else [],
        'labels_l3':     [l3] if l3 else [],
        'vec_l1':        multihot(l1, LABEL_MAP['l1']),
        'vec_l2':        multihot(l2, LABEL_MAP['l2']),
        'vec_l3':        multihot(l3, LABEL_MAP['l3']),
        'is_multilabel': False,
    })

    if (i + 1) % 1000 == 0:
        print(f'  {i+1}/{len(raw)}')

print(f'\nHợp lệ : {len(articles):,}')
print(f'Bỏ qua : {skipped:,}')


Loaded 5,546 bài thô
  1000/5546
  2000/5546
  3000/5546
  4000/5546
  5000/5546

Hợp lệ : 5,541
Bỏ qua : 5


## Cell 6 — Build vocab → lưu file


In [6]:
# Build vocab
counter = Counter(t for a in articles for t in a['tokens'])
vocab   = {'<PAD>': 0, '<UNK>': 1}
for word, cnt in counter.most_common():
    if cnt >= VOCAB_MIN_COUNT:
        vocab[word] = len(vocab)
print(f'Vocab: {len(vocab):,} từ (min_count={VOCAB_MIN_COUNT})')

# Lưu
with open(OUT_DATASET,   'w', encoding='utf-8') as f: json.dump(articles,  f, ensure_ascii=False, indent=2)
with open(OUT_VOCAB,     'w', encoding='utf-8') as f: json.dump(vocab,     f, ensure_ascii=False, indent=2)
with open(OUT_LABEL_MAP, 'w', encoding='utf-8') as f: json.dump(LABEL_MAP, f, ensure_ascii=False, indent=2)

print(f'\n✓ {OUT_DATASET}')
print(f'✓ {OUT_VOCAB}')
print(f'✓ {OUT_LABEL_MAP}')


Vocab: 29,023 từ (min_count=3)

✓ C:\Users\Admin\Documents\nlp\harnn_project\data\process_data\dataset.json
✓ C:\Users\Admin\Documents\nlp\harnn_project\data\process_data\vocab.json
✓ C:\Users\Admin\Documents\nlp\harnn_project\data\process_data\label_map.json


## Cell 7 — Kiểm tra kết quả


In [7]:
total   = len(articles)
has_l2  = sum(1 for a in articles if a['labels_l2'])
has_l3  = sum(1 for a in articles if a['labels_l3'])

print(f'Tổng bài   : {total:,}')
print(f'Có L2      : {has_l2:,} ({has_l2/total*100:.1f}%)')
print(f'Có L3      : {has_l3:,} ({has_l3/total*100:.1f}%)')
print(f'Vocab size : {len(vocab):,}')
print(f'vec_l1 dim : {len(articles[0]["vec_l1"])}')
print(f'vec_l2 dim : {len(articles[0]["vec_l2"])}')
print(f'vec_l3 dim : {len(articles[0]["vec_l3"])}')

# Phân phối L1
l1_dist = Counter(a['labels_l1'][0] for a in articles)
print(f'\nPhân phối L1:')
mx = max(l1_dist.values())
for lb, cnt in sorted(l1_dist.items(), key=lambda x: -x[1]):
    bar = '█' * (cnt * 30 // mx)
    print(f'  {lb:<22} {cnt:>5}  {bar}')

# 3 bài mẫu
print('\n3 bài mẫu:')
for a in articles[:3]:
    print(f"  {a['title'][:55]}")
    print(f"  L1={a['labels_l1']}  L2={a['labels_l2']}  L3={a['labels_l3']}")
    print(f"  tokens={a['tokens'][:6]}  vec_l1 sum={sum(a['vec_l1'])}")
    print()


Tổng bài   : 5,541
Có L2      : 3,505 (63.3%)
Có L3      : 629 (11.4%)
Vocab size : 29,023
vec_l1 dim : 13
vec_l2 dim : 47
vec_l3 dim : 19

Phân phối L1:
  Thể thao                 624  ██████████████████████████████
  Thời sự                  590  ████████████████████████████
  Sức khỏe                 468  ██████████████████████
  Đời sống                 465  ██████████████████████
  Giáo dục                 464  ██████████████████████
  Giải trí                 460  ██████████████████████
  Du lịch                  458  ██████████████████████
  Pháp luật                438  █████████████████████
  Số hóa                   433  ████████████████████
  Thế giới                 341  ████████████████
  Xe                       338  ████████████████
  Khoa học                 289  █████████████
  Kinh doanh               173  ████████

3 bài mẫu:
  Cứu được người đuối nước khi livestream
  L1=['Thời sự']  L2=[]  L3=[]
  tokens=['cứu', 'đuối', 'livestream', 'hà_tuổi', 'trú', 'thôn']  vec_